<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/scraping/backlinks_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔎 Backlink URL Checker — Google Colab Edition

**Features:**
- Expired Domain Detection  
- HTTP Status Code Checker  
- Illegal Content Detector  

---

**Project Link:**  
https://claude.ai/chat/54223600-6d4d-48b7-a8b6-8c5d4ffa4965

**Google Collab**
https://colab.research.google.com/drive/1vzfZlQeKkBZoIoVGmcAkDBPtuWMvR3RG#scrollTo=MHV040LfKWR_

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║         BACKLINK URL CHECKER — Google Colab Edition          ║
# ║   Expired Domain | Status Code | Illegal Content Detector    ║
# ╚══════════════════════════════════════════════════════════════╝
# https://claude.ai/chat/54223600-6d4d-48b7-a8b6-8c5d4ffa4965

# ── CELL 1: Install Dependencies ─────────────────────────────────
# !pip install requests beautifulsoup4 pandas tqdm -q

# ─────────────────────────────────────────────────────────────────
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import re, time, io, sys

# ══════════════════════════════════════════════════════════════════
# LOGGER — terminal-style output dengan warna & emoji
# ══════════════════════════════════════════════════════════════════

class Logger:
    COLORS = {
        "reset":   "\033[0m",
        "bold":    "\033[1m",
        "grey":    "\033[90m",
        "green":   "\033[92m",
        "yellow":  "\033[93m",
        "red":     "\033[91m",
        "cyan":    "\033[96m",
        "magenta": "\033[95m",
        "blue":    "\033[94m",
        "white":   "\033[97m",
    }

    def _ts(self):
        return datetime.now().strftime("%H:%M:%S")

    def _c(self, text, *colors):
        codes = "".join(self.COLORS.get(c, "") for c in colors)
        return f"{codes}{text}{self.COLORS['reset']}"

    def banner(self):
        print(self._c("━" * 62, "cyan"))
        print(self._c("  ██╗   ██╗██████╗ ██╗      ██████╗██╗  ██╗███████╗ ██████╗", "cyan", "bold"))
        print(self._c("  ██║   ██║██╔══██╗██║     ██╔════╝██║  ██║██╔════╝██╔════╝", "cyan", "bold"))
        print(self._c("  ██║   ██║██████╔╝██║     ██║     ███████║█████╗  ██║     ", "cyan", "bold"))
        print(self._c("  ██║   ██║██╔══██╗██║     ██║     ██╔══██║██╔══╝  ██║     ", "cyan", "bold"))
        print(self._c("  ╚██████╔╝██║  ██║███████╗╚██████╗██║  ██║███████╗╚██████╗", "cyan", "bold"))
        print(self._c("   ╚═════╝ ╚═╝  ╚═╝╚══════╝ ╚═════╝╚═╝  ╚═╝╚══════╝ ╚═════╝", "cyan", "bold"))
        print(self._c("  Backlink URL Checker  •  Expired | Status | Illegal", "grey"))
        print(self._c("━" * 62, "cyan"))

    def info(self, msg):
        print(f"  {self._c(self._ts(), 'grey')}  {self._c('INFO', 'cyan', 'bold')}  {msg}")

    def success(self, msg):
        print(f"  {self._c(self._ts(), 'grey')}  {self._c('  OK ', 'green', 'bold')}  {msg}")

    def warn(self, msg):
        print(f"  {self._c(self._ts(), 'grey')}  {self._c('WARN', 'yellow', 'bold')}  {msg}")

    def error(self, msg):
        print(f"  {self._c(self._ts(), 'grey')}  {self._c(' ERR', 'red', 'bold')}  {msg}")

    def result(self, url, status, expired, illegal, err=None):
        short = url[:45] + "…" if len(url) > 46 else url.ljust(46)

        # status code color
        if status == 200:
            s = self._c(f"[{status}]", "green", "bold")
        elif status and 300 <= status < 400:
            s = self._c(f"[{status}]", "yellow", "bold")
        elif status:
            s = self._c(f"[{status}]", "red", "bold")
        else:
            s = self._c("[---]", "grey")

        exp = self._c("⚠ EXPIRED", "yellow") if expired == "Yes" else self._c("✓ active ", "grey")
        ill = self._c("🚫 ILLEGAL", "red", "bold") if illegal == "Yes" else self._c("✓ clean  ", "grey")
        er  = self._c(f"  ⚡ {err}", "red") if err else ""

        print(f"  {self._c(self._ts(), 'grey')}  {s}  {short}  {exp}  {ill}{er}")

    def section(self, title):
        print()
        print(f"  {self._c('┌─', 'cyan')} {self._c(title, 'white', 'bold')} {self._c('─' * (55 - len(title)), 'cyan')}")

    def summary(self, total, ok200, expired, illegal, errors, duration):
        print()
        print(self._c("  ┌" + "─" * 58 + "┐", "cyan"))
        print(self._c("  │", "cyan") + self._c("  📊  HASIL AKHIR PEMERIKSAAN".center(58), "white", "bold") + self._c("│", "cyan"))
        print(self._c("  ├" + "─" * 58 + "┤", "cyan"))

        rows = [
            ("🔗  Total URL diperiksa", str(total),       "white"),
            ("✅  Status 200",          str(ok200),        "green"),
            ("⚠️   Expired Domain",      str(expired),      "yellow"),
            ("🚫  Konten Illegal",       str(illegal),      "red"),
            ("💥  Error / Timeout",      str(errors),       "red"),
            ("⏱️   Durasi",              f"{duration:.1f}s","cyan"),
        ]
        for label, val, color in rows:
            line = f"  {label:<30}  {self._c(val, color, 'bold')}"
            pad = 58 - len(f"  {label:<30}  {val}")
            print(self._c("  │", "cyan") + line + " " * pad + self._c("│", "cyan"))

        print(self._c("  └" + "─" * 58 + "┘", "cyan"))

log = Logger()


# ══════════════════════════════════════════════════════════════════
# KONFIGURASI
# ══════════════════════════════════════════════════════════════════

TIMEOUT = 10

EXPIRED_DOMAIN_INDICATORS = [
    "mamma.com", "sedo.com", "sedoparking.com", "dan.com",
    "hugedomains.com", "godaddy.com/domain", "afternic.com",
    "namecheap.com/domains/registration", "parkingcrew.net",
    "bodis.com", "above.com", "trellian.com", "domainsponsor.com",
    "domainadvert.com", "parked",
]

ILLEGAL_TERMS_ID = [
    "judi", "togel", "slot", "poker", "bandar", "taruhan", "betting",
    "kasino", "casino", "gacor", "maxwin", "scatter", "jackpot",
    "bokep", "ngentot", "memek", "kontol", "bugil", "telanjang",
    "dewasa", "xxx",
]

ILLEGAL_TERMS_EN = [
    "gambling", "bet", "casino", "poker", "lottery", "slot machine",
    "porn", "pornography", "nude", "naked", "sex", "xxx",
    "adult content", "escort",
]

ALL_ILLEGAL_TERMS = ILLEGAL_TERMS_ID + ILLEGAL_TERMS_EN


# ══════════════════════════════════════════════════════════════════
# FUNGSI CEK URL
# ══════════════════════════════════════════════════════════════════

def check_url(url):
    result = {
        "url": url,
        "status_code": None,
        "final_url": None,
        "expired_domain": "No",
        "illegal_terms_found": [],
        "is_illegal": "No",
        "error": None,
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    try:
        resp = requests.get(url, timeout=TIMEOUT, headers=headers, allow_redirects=True)
        result["status_code"] = resp.status_code
        result["final_url"]   = resp.url

        final_url_lower  = resp.url.lower()
        page_text_lower  = resp.text.lower()

        # Cek Expired
        expired_hints = [
            any(ind in final_url_lower for ind in EXPIRED_DOMAIN_INDICATORS),
            "this domain" in page_text_lower and ("sale" in page_text_lower or "parked" in page_text_lower),
            "domain for sale"           in page_text_lower,
            "buy this domain"           in page_text_lower,
            "domain is for sale"        in page_text_lower,
            "this domain may be for sale" in page_text_lower,
        ]
        if any(expired_hints):
            result["expired_domain"] = "Yes"

        # Cek Illegal Terms
        try:
            soup = BeautifulSoup(resp.text, "html.parser")
            for tag in soup(["script", "style", "meta", "head"]):
                tag.decompose()
            clean_text = soup.get_text(separator=" ").lower()
        except Exception:
            clean_text = page_text_lower

        found_terms = [
            term for term in ALL_ILLEGAL_TERMS
            if re.search(r'\b' + re.escape(term) + r'\b', clean_text)
        ]
        if found_terms:
            result["illegal_terms_found"] = found_terms
            result["is_illegal"]          = "Yes"

    except requests.exceptions.Timeout:
        result["error"] = "Timeout"
    except requests.exceptions.ConnectionError:
        result["error"] = "Connection Error"
    except requests.exceptions.TooManyRedirects:
        result["error"] = "Too Many Redirects"
        result["expired_domain"] = "Possible"
    except Exception as e:
        result["error"] = str(e)[:60]

    return result


# ══════════════════════════════════════════════════════════════════
# LOAD CSV
# ══════════════════════════════════════════════════════════════════

def load_urls_from_csv(content_bytes):
    df = pd.read_csv(io.BytesIO(content_bytes), header=0)
    url_col = next(
        (col for col in df.columns if "url" in col.strip().lower()), None
    )
    if url_col is None:
        raise ValueError(f"Kolom URL tidak ditemukan. Kolom tersedia: {list(df.columns)}")
    log.success(f"Kolom URL terdeteksi  →  '{url_col}'")
    urls = df[url_col].dropna().astype(str).str.strip().tolist()
    log.info(f"Total URL dimuat      →  {len(urls)}")
    return urls


# ══════════════════════════════════════════════════════════════════
# RUNNER
# ══════════════════════════════════════════════════════════════════

def run_checker(url_list, delay=0.5):
    results = []
    log.section("MEMULAI PEMERIKSAAN URL")
    print()

    for url in tqdm(url_list, desc="  Progress", unit="url",
                    bar_format="  {l_bar}{bar:30}{r_bar}"):
        res = check_url(url)
        log.result(
            url     = res["url"],
            status  = res["status_code"],
            expired = res["expired_domain"],
            illegal = res["is_illegal"],
            err     = res["error"],
        )
        results.append(res)
        time.sleep(delay)

    df = pd.DataFrame(results)
    df["illegal_terms_found"] = df["illegal_terms_found"].apply(
        lambda x: ", ".join(x) if x else "-"
    )
    return df


# ══════════════════════════════════════════════════════════════════
# FORM UPLOAD — WIDGET COLAB
# ══════════════════════════════════════════════════════════════════

def build_upload_form():
    clear_output()
    log.banner()

    upload_widget = widgets.FileUpload(
        accept       = ".csv",
        multiple     = False,
        description  = "📂 Upload CSV",
        button_style = "primary",
        layout       = widgets.Layout(width="200px"),
    )

    delay_slider = widgets.FloatSlider(
        value       = 0.5, min=0.1, max=3.0, step=0.1,
        description = "Delay (s):",
        style       = {"description_width": "80px"},
        layout      = widgets.Layout(width="350px"),
    )

    run_btn = widgets.Button(
        description  = "  🚀  Jalankan Checker",
        button_style = "success",
        layout       = widgets.Layout(width="220px", height="38px"),
        style        = {"font_weight": "bold"},
    )

    status_label = widgets.Label(value="⬆️  Upload file CSV dulu, lalu klik Jalankan.")
    output_area  = widgets.Output()

    form = widgets.VBox([
        widgets.HTML("<hr style='border-color:#444'>"),
        widgets.HTML("<b style='font-size:14px'>⚙️  Konfigurasi</b>"),
        upload_widget,
        delay_slider,
        widgets.HTML("<br>"),
        run_btn,
        status_label,
        widgets.HTML("<hr style='border-color:#444'>"),
        output_area,
    ], layout=widgets.Layout(padding="10px"))

    display(form)

    def on_run_clicked(_):
        with output_area:
            clear_output()

            if not upload_widget.value:
                log.error("Belum ada file yang diupload!")
                return

            uploaded  = list(upload_widget.value.values())[0]
            filename  = list(upload_widget.value.keys())[0]
            status_label.value = f"⏳  Memproses {filename} ..."

            try:
                log.section(f"FILE  →  {filename}")
                url_list = load_urls_from_csv(uploaded["content"])
            except Exception as e:
                log.error(str(e))
                status_label.value = "❌  Gagal membaca CSV."
                return

            start     = time.time()
            df_result = run_checker(url_list, delay=delay_slider.value)
            duration  = time.time() - start

            log.summary(
                total    = len(df_result),
                ok200    = (df_result["status_code"] == 200).sum(),
                expired  = (df_result["expired_domain"] == "Yes").sum(),
                illegal  = (df_result["is_illegal"] == "Yes").sum(),
                errors   = df_result["error"].notna().sum(),
                duration = duration,
            )

            today           = datetime.now().strftime("%Y%m%d")
            output_filename = f"backlink_filter_{today}.csv"
            df_result.to_csv(output_filename, index=False)
            log.success(f"File disimpan  →  {output_filename}")

            try:
                from google.colab import files
                files.download(output_filename)
                log.success("Auto-download dimulai ✓")
            except Exception:
                log.warn("Auto-download tidak tersedia. Ambil manual dari panel file Colab.")

            status_label.value = f"✅  Selesai! {len(df_result)} URL diperiksa dalam {duration:.1f}s"

    run_btn.on_click(on_run_clicked)


# ══════════════════════════════════════════════════════════════════
# JALANKAN
# ══════════════════════════════════════════════════════════════════
build_upload_form()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ██╗   ██╗██████╗ ██╗      ██████╗██╗  ██╗███████╗ ██████╗
  ██║   ██║██╔══██╗██║     ██╔════╝██║  ██║██╔════╝██╔════╝
  ██║   ██║██████╔╝██║     ██║     ███████║█████╗  ██║     
  ██║   ██║██╔══██╗██║     ██║     ██╔══██║██╔══╝  ██║     
  ╚██████╔╝██║  ██║███████╗╚██████╗██║  ██║███████╗╚██████╗
   ╚═════╝ ╚═╝  ╚═╝╚══════╝ ╚═════╝╚═╝  ╚═╝╚══════╝ ╚═════╝
  Backlink URL Checker  •  Expired | Status | Illegal
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
